In [2]:
print("Installing Apache Airflow... (This may take around 30-45 seconds)")
!pip install apache-airflow > /dev/null 2>&1
print("✔ Apache Airflow installed successfully!\n")

Installing Apache Airflow... (This may take around 30-45 seconds)
✔ Apache Airflow installed successfully!



In [11]:
import os
import csv
from datetime import datetime
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator

def create_orders():
    # Note: Exercise 10 asks for orders.csv without an explicit folder prefix, but let's keep it safe in /tmp/
    file_path = "/tmp/orders.csv"
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    content = """product,quantity,price
Laptop,1,70000
Mouse,4,500
Monitor,2,12000
Keyboard,3,1500"""
    with open(file_path, "w") as f:
        f.write(content.strip())
    print(f"✔ Created {file_path}")

def calculate_order_value():
    file_path = "/tmp/orders.csv"
    total_revenue = 0
    highest_revenue = 0
    best_product = ""

    with open(file_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rev = int(row['quantity']) * int(row['price'])
            total_revenue += rev
            if rev > highest_revenue:
                highest_revenue = rev
                best_product = row['product']

    return total_revenue, best_product

def generate_sales_report():
    total_rev, top_prod = calculate_order_value()
    report_path = "/tmp/sales_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Total Revenue = {total_rev}\nHighest Selling Product = {top_prod}\n")
    print(f"✔ Generated Report at {report_path}")
    print("\n📋 File Content:")
    with open(report_path, "r") as f: print(f.read())

with DAG(dag_id='exercise_10_orders', start_date=datetime(2026, 1, 1), schedule=None, catchup=False) as dag:
    t1 = PythonOperator(task_id='create_orders', python_callable=create_orders)
    t2 = PythonOperator(task_id='calculate_order_value', python_callable=calculate_order_value)
    t3 = PythonOperator(task_id='generate_sales_report', python_callable=generate_sales_report)
    t1 >> t2 >> t3

# Trigger in Colab
create_orders()
generate_sales_report()

✔ Created /tmp/orders.csv
✔ Generated Report at /tmp/sales_report.txt

📋 File Content:
Total Revenue = 100500
Highest Selling Product = Laptop

